# θ 그룹 크기별 Adaptive Quantum COBYLA 비교

기존 `adaptive_Q_cobyla_vs_Q_cobyla.ipynb`와 **동일한 모델 구조·규모·시드·데이터·초기화·Round·손실 평가 예산**을 사용합니다. 달라지는 변수는 `theta_group_size`입니다.

- `1`: 파라미터별 θ
- `16`, `128`: 그룹별 θ
- `None`: 전체 파라미터가 θ 하나 공유

Qiskit statevector 시뮬레이션이며 실제 양자 하드웨어의 우월성을 입증하지 않습니다.
추가 진단: Round별 θ 이동량, +1 확률의 분산, 그룹 크기 1 대비 실제 방향 샘플의 불일치 비율을 기록합니다.


In [ ]:
import math
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from modules.optimizers.AdaptiveQuantumDirectionCOBYLA import AdaptiveQuantumDirectionCOBYLA

SEEDS = [11, 23, 37, 53, 71]
ROUNDS = 8
EVALS_PER_ROUND = 12
MAX_PARAMS = 20000
RUN = [("linear", "S"), ("mlp", "S"), ("residual", "S"), ("transformer", "S")]
SCALES = {"S": (8, 1), "M": (64, 2), "L": (256, 6), "XL": (1024, 24)}
GROUP_SIZES = [1, 16, 128, None]
torch.set_num_threads(min(4, torch.get_num_threads()))


In [ ]:
class Residual(nn.Module):
    def __init__(self, w, d):
        super().__init__()
        self.inp = nn.Linear(8, w)
        self.blocks = nn.ModuleList([
            nn.Sequential(nn.Linear(w, w), nn.GELU(), nn.Linear(w, w))
            for _ in range(d)
        ])
        self.out = nn.Linear(w, 1)

    def forward(self, x):
        z = self.inp(x)
        for block in self.blocks:
            z = z + block(z) / math.sqrt(z.shape[-1])
        return self.out(z)


class Transformer(nn.Module):
    def __init__(self, w, d):
        super().__init__()
        self.embed = nn.Linear(1, w)
        layer = nn.TransformerEncoderLayer(
            d_model=w, nhead=2, dim_feedforward=4 * w,
            dropout=0.0, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=d, enable_nested_tensor=False,
        )
        self.out = nn.Linear(w, 1)

    def forward(self, x):
        return self.out(self.encoder(self.embed(x.unsqueeze(-1))).mean(1))


def model_for(family, scale):
    w, d = SCALES[scale]
    if family == "linear":
        return nn.Linear(8, 1)
    if family == "mlp":
        layers = [nn.Linear(8, w), nn.Tanh()]
        for _ in range(d - 1):
            layers.extend([nn.Linear(w, w), nn.Tanh()])
        return nn.Sequential(*layers, nn.Linear(w, 1))
    if family == "residual":
        return Residual(w, d)
    if family == "transformer":
        return Transformer(w, d)
    raise ValueError(family)


def nparams(model):
    return sum(p.numel() for p in model.parameters())


def dataset(seed):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(256, 8)).astype("float32")
    y = (
        0.7 * np.sin(x[:, 0]) + 0.4 * x[:, 1] ** 2
        - 0.5 * x[:, 2] * x[:, 3] + 0.2 * x[:, 4]
        + rng.normal(0, 0.05, 256)
    ).astype("float32")
    order = rng.permutation(256)
    return [
        torch.from_numpy(v)
        for v in (
            x[order[:192]], y[order[:192], None],
            x[order[192:]], y[order[192:], None],
        )
    ]


catalog = pd.DataFrame([
    {"family": family, "scale": scale, "parameters": nparams(model_for(family, scale))}
    for family in ("linear", "mlp", "residual", "transformer")
    for scale in SCALES
])
catalog["runnable"] = catalog["parameters"] <= MAX_PARAMS
display(catalog)


## 실험 실행

기존 Notebook과 동일하게 Linear·MLP·Residual·Transformer의 **S 규모**, 시드 5개, Round 8회, Round당 COBYLA 평가 예산 12회를 사용합니다. 각 시드에서 모델 초기화와 데이터가 동일하도록 고정합니다. 그룹 크기별로 별도의 최적화기를 생성합니다.

In [ ]:
class DirectionRecordingAdaptiveCOBYLA(AdaptiveQuantumDirectionCOBYLA):
    """Record sampled signs before COBYLA changes the direction scale."""
    def create_directions(self, count):
        directions = super().create_directions(count)
        self.last_sampled_signs = np.concatenate([
            np.sign(direction.detach().cpu().reshape(-1).numpy()).astype(np.int8)
            for direction in directions
        ])
        return directions


def run_one(family, scale, seed, group_size):
    torch.manual_seed(seed + 1000)
    model = model_for(family, scale)
    if nparams(model) > MAX_PARAMS:
        raise ValueError("parameter limit exceeded")
    xt, yt, xv, yv = dataset(seed)
    optimizer = DirectionRecordingAdaptiveCOBYLA(
        steps=1, direction_cobyla_evaluations=EVALS_PER_ROUND,
        adaptive_learning_rate=0.1, theta_group_size=group_size,
        cobyla_radius=0.1, seed=seed,
    )
    optimizer.prepare(model)
    calls = 0

    def objective():
        nonlocal calls
        calls += 1
        with torch.no_grad():
            return float(nn.functional.mse_loss(model(xt), yt))

    def mse(x, y):
        with torch.no_grad():
            return float(nn.functional.mse_loss(model(x), y))

    def theta_diagnostics(round_index):
        theta = np.asarray(optimizer.sampler.theta, dtype=float)
        probabilities = optimizer.sampler.probabilities_plus
        return {
            "round": round_index,
            "mean_abs_theta_shift": float(np.mean(np.abs(theta - np.pi / 2))),
            "max_abs_theta_shift": float(np.max(np.abs(theta - np.pi / 2))),
            "std_probability_plus": float(np.std(probabilities)),
            "mean_probability_plus": float(np.mean(probabilities)),
        }

    history = [{"round": 0, "train_mse": mse(xt, yt), "val_mse": mse(xv, yv)}]
    theta_history = [theta_diagnostics(0)]
    direction_history = []
    start = time.perf_counter()
    for round_index in range(1, ROUNDS + 1):
        before = calls
        info = optimizer.optimize(objective)
        assert calls - before == info["optimization_evaluations"]
        assert calls - before <= EVALS_PER_ROUND
        direction_history.append({
            "round": round_index,
            "signs": optimizer.last_sampled_signs.copy(),
        })
        theta_history.append(theta_diagnostics(round_index))
        history.append({
            "round": round_index, "train_mse": mse(xt, yt),
            "val_mse": mse(xv, yv),
        })
    return {
        "family": family, "scale": scale, "seed": seed,
        "theta_group_size": group_size,
        "label": "shared" if group_size is None else str(group_size),
        "parameters": nparams(model),
        "theta_count": len(optimizer.sampler.theta),
        "objective_calls": calls,
        "elapsed_seconds": time.perf_counter() - start,
        "history": history,
        "theta_history": theta_history,
        "direction_history": direction_history,
    }


results = []
for family, scale in RUN:
    for seed in SEEDS:
        for group_size in GROUP_SIZES:
            result = run_one(family, scale, seed, group_size)
            results.append(result)
            print(
                family, scale, seed, result["label"],
                round(result["history"][-1]["val_mse"], 6),
            )

df = pd.DataFrame([
    {k: v for k, v in result.items()
     if k not in ("history", "theta_history", "direction_history")}
    | {
        "initial_val_mse": result["history"][0]["val_mse"],
        "final_val_mse": result["history"][-1]["val_mse"],
        "val_mse_reduction": (
            result["history"][0]["val_mse"] - result["history"][-1]["val_mse"]
        ),
    }
    for result in results
])
display(df.sort_values(["family", "scale", "seed", "theta_count"]))


## 그룹 크기별 요약 및 시드별 짝 비교

`diff_vs_parameter` = 해당 그룹 최종 검증 MSE − 파라미터별(그룹 크기 1) 최종 검증 MSE. 음수면 해당 그룹 설정이 더 낮은 MSE입니다.

In [ ]:
summary = df.groupby(["family", "scale", "label"], sort=False).agg(
    runs=("seed", "count"),
    theta_count=("theta_count", "first"),
    mean_final_val_mse=("final_val_mse", "mean"),
    mean_val_mse_reduction=("val_mse_reduction", "mean"),
    mean_seconds=("elapsed_seconds", "mean"),
    mean_objective_calls=("objective_calls", "mean"),
).reset_index()
display(summary)

paired = df.pivot(
    index=["family", "scale", "seed"],
    columns="label",
    values="final_val_mse",
)
diff = paired.subtract(paired["1"], axis=0)
diff.columns = [f"diff_vs_parameter_{label}" for label in diff.columns]
display(diff.reset_index())


## Round별 검증 MSE

동일 시드·Round에서 그룹 크기별 검증 MSE를 비교합니다.

In [ ]:
round_rows = [
    {
        "family": result["family"], "scale": result["scale"],
        "seed": result["seed"], "label": result["label"],
        "round": point["round"], "train_mse": point["train_mse"],
        "val_mse": point["val_mse"],
    }
    for result in results for point in result["history"]
]
round_df = pd.DataFrame(round_rows)
display(round_df.pivot(
    index=["family", "scale", "seed", "round"],
    columns="label", values="val_mse",
).reset_index())
display(round_df.groupby(
    ["family", "scale", "label", "round"], sort=False,
)["val_mse"].mean().unstack("label"))


## 해석상 한계

- 모델 종류, 규모 S, 시드, 데이터 생성, 모델 초기화, Round, COBYLA 평가 예산은 기존 비교 Notebook과 동일합니다.
- 시드 5개와 합성 데이터에 대한 탐색적 비교로, 통계적 우위를 단정할 수 없습니다.
- 그룹별 θ는 방향 확률만 공유하며 개별 파라미터의 손실 기여도를 추정하지 않습니다.
- 현재 구현은 파라미터마다 Qiskit 회로를 생성하므로 θ 개수가 줄어도 실행 시간이 비례해서 감소하지 않을 수 있습니다.
- 실행 결과는 Notebook을 실행하기 전까지 생성되지 않습니다.
- 방향 비교용 ±1 배열은 실행 중 메모리에 보관합니다. 대규모 모델에서는 저장 비용이 증가합니다.


## θ 변화량 및 실제 방향 샘플 비교

각 Round 종료 후 θ가 초기값 π/2에서 얼마나 이동했는지 기록합니다. `std_probability_plus`는 그룹별 +1 확률의 표준편차입니다. 그룹이 하나인 설정에서는 정의상 0입니다.

`direction_change_rate`는 **같은 모델·규모·시드·Round**에서 그룹 크기 1과 비교했을 때 실제 샘플링된 ±1 방향이 서로 다른 파라미터의 비율(Hamming distance / 파라미터 수)입니다. 이 값은 COBYLA가 선택한 최종 이동 방향이 아니라 **샘플러의 원래 방향**을 비교합니다. 샘플링 시드는 동일하지만 그룹별 확률이 달라질 수 있습니다.

In [ ]:
theta_rows = [
    {"family": result["family"], "scale": result["scale"],
     "seed": result["seed"], "label": result["label"], **item}
    for result in results for item in result["theta_history"]
]
theta_df = pd.DataFrame(theta_rows)
display(theta_df.sort_values(["family", "scale", "seed", "round", "label"]))
display(theta_df.groupby(["family", "scale", "label", "round"], sort=False)[
    ["mean_abs_theta_shift", "max_abs_theta_shift",
     "std_probability_plus", "mean_probability_plus"]
].mean().reset_index())

reference = {
    (result["family"], result["scale"], result["seed"]): {
        item["round"]: item["signs"]
        for item in result["direction_history"]
    }
    for result in results if result["theta_group_size"] == 1
}
direction_rows = []
for result in results:
    baseline = reference[(result["family"], result["scale"], result["seed"])]
    for item in result["direction_history"]:
        original = baseline[item["round"]]
        current = item["signs"]
        assert original.shape == current.shape
        direction_rows.append({
            "family": result["family"], "scale": result["scale"],
            "seed": result["seed"], "label": result["label"],
            "round": item["round"],
            "direction_change_rate": float(np.mean(current != original)),
            "changed_parameters": int(np.count_nonzero(current != original)),
            "parameters": len(current),
        })
direction_df = pd.DataFrame(direction_rows)
display(direction_df.sort_values(["family", "scale", "seed", "round", "label"]))
display(direction_df.groupby(
    ["family", "scale", "label", "round"], sort=False,
)[["direction_change_rate", "changed_parameters"]].mean().reset_index())
